# Master — REGISTRACIJA: poređenje TIPOVA registracije (kičma)

Za svakog pacijenta poravnava **dodatnu sekvencu → T2**:

| tip | opis |
|---|---|
| bez | samo preuzorkovanje na T2 mrežu (koordinate skenera) |
| rigidna | 6 parametara (rotacija + pomeraj) |
| afina | 12 parametara (+ skaliranje, smicanje) |
| deformabilna | B-spline posle rigidne (savitljiva — kičma je savitljiva struktura) |
| autorska | registracija autora skupa (`preprocessedAndRegistered`) |

Mera poravnanja: **Mattes uzajamna informacija (MI)** — **niža = bolje poravnato**.
Uticaj na **kvalitet detekcije (Dice)** meri se u `Master_kicma_REG_NOREG.ipynb`.



## Ćelija 0 — priprema

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q SimpleITK nibabel

import os, glob, time, csv
import numpy as np, SimpleITK as sitk
MASTER = '/content/drive/MyDrive/Master/kicma'
REZ_DIR = f'{MASTER}/rezultati'; os.makedirs(REZ_DIR, exist_ok=True)

def svi_nii(folder):
    return sorted(glob.glob(f'{folder}/**/*.nii*', recursive=True))
def nadji(folder, *k, ne=()):
    fs = svi_nii(folder)
    for x in k:
        for f in fs:
            n = os.path.basename(f).lower()
            if x in n and not any(y in n for y in ne):
                return f
    return None
def fajlovi(sub_dir):
    """T2, maska i dodatna sekvenca (STIR/PSIR/MP2RAGE) jednog pacijenta."""
    if not os.path.isdir(sub_dir): return dict(t2=None, mask=None, extra=None, seq=None)
    maske = ('lesion', 'mask', 'seg', 'label')
    t2 = nadji(sub_dir, 't2', ne=maske)
    mask = nadji(sub_dir, 'lesionmask', 'lesion', 'mask')
    extra, seq = None, None
    for s in ['stir', 'psir', 'mp2rage', 'unit1']:
        f = nadji(sub_dir, s, ne=maske)
        if f: extra, seq = f, ('MP2RAGE' if s == 'unit1' else s.upper()); break
    return dict(t2=t2, mask=mask, extra=extra, seq=seq)

print('spremno')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
spremno


## Ćelija 1 — raspakivanje

In [ ]:
import tarfile
tar_path = f'{MASTER}/podaci_sirovi/MS_MultiSpine_dataset.tar.gz'
if not os.path.exists('/content/multispine'):
    print('raspakivam (nekoliko min)...')
    with tarfile.open(tar_path) as t: t.extractall('/content/multispine')
der = glob.glob('/content/multispine/**/derivatives', recursive=True)
assert der, 'Nema derivatives foldera u tar-u'
PAR = f'{der[0]}/preprocessedAndRegistered'
PRE = f'{der[0]}/preprocessed'
print('preprocessedAndRegistered:', os.path.isdir(PAR), '| preprocessed:', os.path.isdir(PRE))
SUBS = sorted(s for s in os.listdir(PAR) if s.startswith('sub-'))
print('pacijenata:', len(SUBS))

preprocessedAndRegistered: True | preprocessed: True
pacijenata: 100


## Ćelija 2 — registracije i mera poravnanja

In [ ]:
def centar(img):
    return img.TransformContinuousIndexToPhysicalPoint([(s - 1) / 2.0 for s in img.GetSize()])

def mi(fixed, moving_on_fixed):
    R = sitk.ImageRegistrationMethod()
    R.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    R.SetMetricSamplingStrategy(R.RANDOM); R.SetMetricSamplingPercentage(0.2, seed=42)
    R.SetInterpolator(sitk.sitkLinear)
    R.SetInitialTransform(sitk.Transform(3, sitk.sitkIdentity))
    return R.MetricEvaluate(fixed, moving_on_fixed)

def na_mrezu(moving, fixed, tx):
    return sitk.Resample(moving, fixed, tx, sitk.sitkLinear, 0.0, sitk.sitkFloat32)

def osnova(nivoi=([4, 2, 1], [2, 1, 0])):
    R = sitk.ImageRegistrationMethod()
    R.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    R.SetMetricSamplingStrategy(R.RANDOM); R.SetMetricSamplingPercentage(0.2, seed=42)
    R.SetInterpolator(sitk.sitkLinear)
    R.SetShrinkFactorsPerLevel(nivoi[0]); R.SetSmoothingSigmasPerLevel(nivoi[1])
    return R

def rigidna(f, m):
    tx = sitk.Euler3DTransform(); tx.SetCenter(centar(f))
    R = osnova(); R.SetOptimizerAsRegularStepGradientDescent(0.5, 1e-4, 100, relaxationFactor=0.5)
    R.SetOptimizerScalesFromPhysicalShift(); R.SetInitialTransform(tx, inPlace=False)
    return R.Execute(f, m)

def afina(f, m):
    tx = sitk.AffineTransform(3); tx.SetCenter(centar(f))
    R = osnova(); R.SetOptimizerAsRegularStepGradientDescent(0.5, 1e-4, 100, relaxationFactor=0.5)
    R.SetOptimizerScalesFromPhysicalShift(); R.SetInitialTransform(tx, inPlace=False)
    return R.Execute(f, m)

def deformabilna(f, m, rig):
    bs = sitk.BSplineTransformInitializer(f, [4, 4, 4])
    R = osnova(([4, 2], [2, 1])); R.SetOptimizerAsLBFGSB(gradientConvergenceTolerance=1e-5, numberOfIterations=30)
    R.SetMovingInitialTransform(rig); R.SetInitialTransform(bs, inPlace=True)
    R.Execute(f, m)
    comp = sitk.CompositeTransform(3); comp.AddTransform(rig); comp.AddTransform(bs)
    return comp
print('funkcije spremne')

funkcije spremne


## Ćelija 3 — petlja po pacijentima

In [ ]:
import time
sitk.ProcessObject.SetGlobalDefaultNumberOfThreads(os.cpu_count())

def osnova(nivoi=([4, 2], [2, 1]), uzorak=0.05):
    R = sitk.ImageRegistrationMethod()
    R.SetMetricAsMattesMutualInformation(numberOfHistogramBins=32)
    R.SetMetricSamplingStrategy(R.RANDOM); R.SetMetricSamplingPercentage(uzorak, seed=42)
    R.SetInterpolator(sitk.sitkLinear)
    R.SetShrinkFactorsPerLevel(nivoi[0]); R.SetSmoothingSigmasPerLevel(nivoi[1])
    return R

def rigidna(f, m):
    tx = sitk.Euler3DTransform(); tx.SetCenter(centar(f))
    R = osnova(); R.SetOptimizerAsRegularStepGradientDescent(0.5, 1e-3, 60, relaxationFactor=0.5)
    R.SetOptimizerScalesFromPhysicalShift(); R.SetInitialTransform(tx, inPlace=False)
    return R.Execute(f, m)

def afina(f, m):
    tx = sitk.AffineTransform(3); tx.SetCenter(centar(f))
    R = osnova(); R.SetOptimizerAsRegularStepGradientDescent(0.5, 1e-3, 60, relaxationFactor=0.5)
    R.SetOptimizerScalesFromPhysicalShift(); R.SetInitialTransform(tx, inPlace=False)
    return R.Execute(f, m)

def deformabilna(f, m, rig):
    bs = sitk.BSplineTransformInitializer(f, [3, 3, 3])
    R = osnova(([4], [2])); R.SetOptimizerAsLBFGSB(gradientConvergenceTolerance=1e-4, numberOfIterations=15)
    R.SetMovingInitialTransform(rig); R.SetInitialTransform(bs, inPlace=True)
    R.Execute(f, m)
    comp = sitk.CompositeTransform(3); comp.AddTransform(rig); comp.AddTransform(bs)
    return comp

SUBSET_PO_SEKVENCI = 10
DEFORMABILNA = True

from collections import defaultdict
po_seq = defaultdict(list)
for sub in SUBS:
    q = fajlovi(f'{PRE}/{sub}')['seq'] or fajlovi(f'{PAR}/{sub}')['seq']
    po_seq[q].append(sub)
skup = [s for q in po_seq for s in (po_seq[q] if SUBSET_PO_SEKVENCI is None else po_seq[q][:SUBSET_PO_SEKVENCI])]
print('uzimam', len(skup), 'pacijenata', flush=True)

rows = []; t_start = time.time()
for i, sub in enumerate(skup, 1):
    t_pac = time.time()
    a = fajlovi(f'{PAR}/{sub}'); b = fajlovi(f'{PRE}/{sub}')
    t2 = b['t2'] or a['t2']
    if not (t2 and b['extra']):
        print(f'[{i}] {sub}: nema T2 ili dodatne -> preskacem', flush=True); continue
    fixed = sitk.ReadImage(t2, sitk.sitkFloat32); mov = sitk.ReadImage(b['extra'], sitk.sitkFloat32)
    seq = b['seq'] or a['seq']
    print(f'[{i}/{len(skup)}] {sub} {seq}  T2 {fixed.GetSize()}  dodatna {mov.GetSize()}', flush=True)
    def zapis(tip, img, sek):
        v = mi(fixed, img)
        rows.append(dict(pacijent=sub, sekvenca=seq, tip=tip, MI=v, sekundi=round(sek, 1)))
        print(f'     {tip:<13} MI={v:.3f}  ({sek:.0f}s)', flush=True)
    try:
        zapis('bez', na_mrezu(mov, fixed, sitk.Transform(3, sitk.sitkIdentity)), 0)
        t = time.time(); rig = rigidna(fixed, mov); zapis('rigidna', na_mrezu(mov, fixed, rig), time.time() - t)
        t = time.time(); af = afina(fixed, mov);   zapis('afina', na_mrezu(mov, fixed, af), time.time() - t)
        if DEFORMABILNA:
            t = time.time(); de = deformabilna(fixed, mov, rig); zapis('deformabilna', na_mrezu(mov, fixed, de), time.time() - t)
        if a['extra']:
            zapis('autorska', na_mrezu(sitk.ReadImage(a['extra'], sitk.sitkFloat32), fixed, sitk.Transform(3, sitk.sitkIdentity)), 0)
        print(f'     ukupno {time.time() - t_pac:.0f}s', flush=True)
    except Exception as e:
        print(f'     greska {e}', flush=True)
print(f'\ngotovo za {(time.time() - t_start) / 60:.1f} min')

uzimam 30 pacijenata
[1/30] sub-001 STIR  T2 (600, 600, 90)  dodatna (600, 600, 90)
     bez           MI=-0.325  (0s)
     rigidna       MI=-0.324  (33s)
     afina         MI=-0.318  (32s)
     deformabilna  MI=-0.296  (70s)
     autorska      MI=-0.325  (0s)
     ukupno 218s
[2/30] sub-002 STIR  T2 (760, 760, 96)  dodatna (760, 760, 96)
     bez           MI=-0.231  (0s)
     rigidna       MI=-0.233  (55s)
     afina         MI=-0.230  (56s)
     deformabilna  MI=-0.218  (116s)
     autorska      MI=-0.241  (0s)
     ukupno 368s
[3/30] sub-003 STIR  T2 (755, 755, 99)  dodatna (755, 755, 99)
     bez           MI=-0.216  (0s)
     rigidna       MI=-0.214  (56s)
     afina         MI=-0.213  (58s)
     deformabilna  MI=-0.206  (119s)
     autorska      MI=-0.222  (0s)
     ukupno 385s
[4/30] sub-004 STIR  T2 (700, 700, 101)  dodatna (700, 700, 101)
     bez           MI=-0.237  (0s)
     rigidna       MI=-0.236  (48s)
     afina         MI=-0.225  (52s)
     deformabilna  MI=-0.211  (

## Ćelija 4 — sažetak po tipu i sekvenci + čuvanje

In [ ]:
from collections import defaultdict
mi_bez = {x['pacijent']: x['MI'] for x in rows if x['tip'] == 'bez'}
tipovi = [t for t in ['bez', 'rigidna', 'afina', 'deformabilna', 'autorska'] if any(x['tip'] == t for x in rows)]
sekv = ['SVE', 'STIR', 'PSIR', 'MP2RAGE']
SAZ = []
print("Mattes MI (nize = bolje). Δ = razlika u odnosu na 'bez'; % bolje = udeo pacijenata gde je tip bolji od 'bez'.\n")
print(f"{'tip':<14}{'sekv.':<9}{'n':>4}{'MI':>9}{'Δ':>9}{'% bolje':>9}{'s/pac':>7}")
for t in tipovi:
    for s in sekv:
        sel = [x for x in rows if x['tip'] == t and (s == 'SVE' or x['sekvenca'] == s)]
        if not sel: continue
        d = [x['MI'] - mi_bez[x['pacijent']] for x in sel]
        r_ = dict(tip=t, sekvenca=s, n=len(sel), MI=float(np.mean([x['MI'] for x in sel])),
                  delta=float(np.mean(d)), pct_bolje=float(100 * np.mean([v < 0 for v in d])),
                  sek=float(np.mean([x['sekundi'] for x in sel])))
        SAZ.append(r_)
        print(f"{t:<14}{s:<9}{r_['n']:>4}{r_['MI']:>9.4f}{r_['delta']:>9.4f}{r_['pct_bolje']:>8.0f}%{r_['sek']:>7.1f}")
    print()
from datetime import datetime
ts = datetime.now().strftime('%Y%m%d_%H%M')
for name, data in [(f'registracija_tipovi_po_pacijentu_{ts}.csv', rows), (f'registracija_tipovi_SAZETAK_{ts}.csv', SAZ)]:
    with open(f'{REZ_DIR}/{name}', 'w', newline='') as fh:
        w = csv.DictWriter(fh, fieldnames=list(data[0].keys())); w.writeheader(); w.writerows(data)
    print('sacuvano:', f'{REZ_DIR}/{name}')

Mattes MI (nize = bolje). Δ = razlika u odnosu na 'bez'; % bolje = udeo pacijenata gde je tip bolji od 'bez'.

tip           sekv.       n       MI        Δ  % bolje  s/pac
bez           SVE        30  -0.2014   0.0000       0%    0.0
bez           STIR       10  -0.2761   0.0000       0%    0.0
bez           PSIR       10  -0.1383   0.0000       0%    0.0
bez           MP2RAGE    10  -0.1896   0.0000       0%    0.0

rigidna       SVE        30  -0.1997   0.0016      20%   33.2
rigidna       STIR       10  -0.2760   0.0001      40%   39.4
rigidna       PSIR       10  -0.1382   0.0001      20%   36.2
rigidna       MP2RAGE    10  -0.1850   0.0046       0%   23.8

afina         SVE        30  -0.1965   0.0049       3%   37.4
afina         STIR       10  -0.2709   0.0052      10%   41.2
afina         PSIR       10  -0.1335   0.0048       0%   41.0
afina         MP2RAGE    10  -0.1851   0.0046       0%   30.0

deformabilna  SVE        30  -0.1927   0.0087      30%   71.3
deformabilna  STIR